In [2]:
from single_cellm.jointemb.geneformer_model import (
    GeneformerConfig,
    GeneformerModel,
    GeneformerTranscriptomeProcessor,
)
import torch
from transformers import BioGptConfig, AutoTokenizer
from single_cellm.jointemb.model import (
    TranscriptomeTextDualEncoderConfig,
    TranscriptomeTextDualEncoderModel,
)
from single_cellm.jointemb.processing import TranscriptomeTextDualEncoderProcessor
from pathlib import Path
import subprocess
import yaml

# Initialize BioGpt and Geneformer model (with pretrained weights)
pwd=%pwd
DATASET = "daniel"

PROJECT_DIR = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], cwd=pwd).decode('utf-8').strip())

with open(PROJECT_DIR / "config.yaml") as f:
    config = yaml.safe_load(f)

config_transcriptome = {
    "model_directory": str(PROJECT_DIR / "modules" / "Geneformer" / "geneformer-12L-30M"),#str(PROJECT_DIR / "resources" / "geneformer-12L-30M"),
    "model_type": "geneformer"
}

config_text = {
    "model_type": "biogpt"
}

model_config = TranscriptomeTextDualEncoderConfig(
    projection_dim=512,
    logit_scale_init_value=2.6592,  # from the original CLIP paper. TODO does this work well for us?
    transcriptome_config=config_transcriptome,
    text_config=config_text,
)
device = torch.device("cuda")

model = TranscriptomeTextDualEncoderModel(config=model_config)
# manually load pretrained
model.text_model = model.text_model.from_pretrained("microsoft/biogpt")
_ = model.to(device)
# geneformer is pretrained implicitly (should change this at some point)

/msc/home/mschae83/miniconda3/envs/singlecellm_tmpVS4/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import anndata
import scanpy as sc

#adata = anndata.read_csv(
#    PROJECT_DIR / f"resources/{DATASET}/read_count_table.csv",
#    first_column_names=True,
#).T


adata = anndata.read_h5ad(
    PROJECT_DIR / f"results/{DATASET}/full_data.h5ad"
)

adata

AnnData object with n_obs × n_vars = 926 × 59552
    obs: 'featurecounts_Percent_Assigned_', 'featurecounts_Percent_Duplicates', 'featurecounts_Percent_Assigned_O', 'featurecounts_Percent_Assigned_M', 'featurecounts_Percent_Assigned_MO', 'sra_study_acc', 'bio_project_acc', 'Tissue', 'Tissue_subtype', 'Disease', 'Disease_subtype', 'Treated', 'biotype_transcribed_unprocessed_pseudogene', 'biotype_unprocessed_pseudogene', 'biotype_miRNA', 'biotype_lncRNA', 'biotype_protein_coding', 'biotype_processed_pseudogene', 'biotype_snRNA', 'biotype_transcribed_processed_pseudogene', 'biotype_misc_RNA', 'biotype_TEC', 'biotype_transcribed_unitary_pseudogene', 'biotype_snoRNA', 'biotype_scaRNA', 'biotype_rRNA_pseudogene', 'biotype_unitary_pseudogene', 'biotype_polymorphic_pseudogene', 'biotype_pseudogene', 'biotype_rRNA', 'biotype_IG_V_pseudogene', 'biotype_scRNA', 'biotype_IG_V_gene', 'biotype_IG_C_gene', 'biotype_IG_J_gene', 'biotype_sRNA', 'biotype_ribozyme', 'biotype_translated_processed_pseudoge

In [4]:
adata_filt = adata.copy()
sc.pp.highly_variable_genes(adata_filt)
adata_filt = adata_filt[:,adata_filt.var['highly_variable']==True]
adata_filt.obs.index.name = None#adata_filt.obs.reset_index(drop=True)
adata_filt.X.shape

/msc/home/mschae83/miniconda3/envs/singlecellm_tmpVS4/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:207: RuntimeWarning: invalid value encountered in log
  dispersion = np.log(dispersion)
/msc/home/mschae83/miniconda3/envs/singlecellm_tmpVS4/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:215: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby('mean_bin')['dispersions']
/msc/home/mschae83/miniconda3/envs/singlecellm_tmpVS4/lib/python3.9/site-packages/anndata/_core/anndata.py:1113: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if not is_categorical_dtype(df_full[k]):


(926, 4758)

In [5]:
adata_filt.obs

,featurecounts_Percent_Assigned_,featurecounts_Percent_Duplicates,featurecounts_Percent_Assigned_O,featurecounts_Percent_Assigned_M,featurecounts_Percent_Assigned_MO,sra_study_acc,bio_project_acc,Tissue,Tissue_subtype,Disease,...,star_number_unmapped_reads_too_many_mismatches,star_percent_unmapped_reads_too_many_mismatches,star_number_unmapped_reads_too_short,star_percent_unmapped_reads_too_short,star_number_unmapped_reads_other,star_percent_unmapped_reads_other,star_number_chimeric_reads,star_percent_chimeric_reads,has_salmon_quantification,natural_language_annotation
SRX085333,3.481,9.185,3.950,7.464,8.368,SRP007584,Rob_Research_Fusobacterium,Bowel,Colon,Colorectal cancer,...,0.0,0.0,115203.0,8.77,56325.0,4.29,0.0,0.0,False,This sample is from a primary colorectal carci...
SRX114777,1.549,9.412,1.757,9.176,10.289,SRP010181,Rob_Research_Colon,Bowel,Colon,Colorectal cancer,...,0.0,0.0,293929.0,8.42,74072.0,2.12,0.0,0.0,False,This sample is from a primary colorectal cance...
SRX099138,4.924,17.525,5.334,67.518,74.198,SRP008496,GSE32424,Esophagus,NaN,Healthy,...,0.0,0.0,869939.0,11.28,807538.0,10.47,0.0,0.0,False,"This sample is from a healthy individual, spec..."
SRX081995,18.465,41.186,20.292,31.328,40.305,SRP007412,GSE30352,Kidney,NaN,Healthy,...,0.0,0.0,2628838.0,10.33,1088758.0,4.28,0.0,0.0,False,This sample is from a healthy male kidney. It ...
SRX083168,19.856,46.335,21.700,28.790,32.158,SRP007483,GSE30573,Brain,Frontal cortex,Mental disorder,...,0.0,0.0,257888.0,0.56,136300.0,0.30,0.0,0.0,False,This sample is a primary sample of postmortem ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRX7633544,27.022,25.815,29.035,29.934,32.390,SRP245232,GSE144259,Bowel,Colon,Colorectal cancer,...,0.0,0.0,316381.0,1.00,42302.0,0.13,0.0,0.0,False,This sample is from a primary colon cancer tis...
SRX7646068,0.107,45.648,0.115,7.949,8.450,SRP245817,GSE144441,Skin,NaN,Healthy,...,0.0,0.0,10868192.0,72.09,4736.0,0.03,0.0,0.0,False,"The sample is from a healthy, non-diabetic 69-..."
SRX8297296,11.534,41.123,12.538,34.994,47.236,SRP192421,GSE129752,Blood,NaN,Antineutrophil cytoplasmic antibody associated...,...,0.0,0.0,31230.0,0.47,2428.0,0.04,0.0,0.0,False,This sample is derived from the whole blood of...
SRX8297304,21.435,36.049,23.421,40.759,47.885,SRP192421,GSE129752,Blood,NaN,Antineutrophil cytoplasmic antibody associated...,...,0.0,0.0,47139.0,0.83,5675.0,0.10,0.0,0.0,False,This sample is derived from the whole blood of...


In [6]:
import pandas as pd
if DATASET == "immgen":
    annot_metdat_df = pd.Series(annot.keys()).str.split('#').str[0].str.split('.',expand=True)
    annot_metdat_df = pd.concat([annot_metdat_df,
                                 pd.Series(annot).reset_index()],
                                axis=1
                               )
    annot_metdat_df.columns = [
        'CellType0','CellType1',
        'CellType2','CellType3',
        'CombdKey','Text'
    ]
    display(annot_metdat_df)
    
elif DATASET == "daniel":
    cols_oi = ['sra_study_acc', 'bio_project_acc',
               'Tissue', 'Tissue_subtype',
               'Disease', 'Disease_subtype', 'Treated'] # 'natural_language_annotation'
    annot_metdat_df = adata.obs[cols_oi]
    display(annot_metdat_df)


,sra_study_acc,bio_project_acc,Tissue,Tissue_subtype,Disease,Disease_subtype,Treated
SRX085333,SRP007584,Rob_Research_Fusobacterium,Bowel,Colon,Colorectal cancer,Colorectal carcinoma,0.0
SRX114777,SRP010181,Rob_Research_Colon,Bowel,Colon,Colorectal cancer,NaN,0.0
SRX099138,SRP008496,GSE32424,Esophagus,NaN,Healthy,Adjacent nonmalignant,0.0
SRX081995,SRP007412,GSE30352,Kidney,NaN,Healthy,NaN,0.0
SRX083168,SRP007483,GSE30573,Brain,Frontal cortex,Mental disorder,Autism spectrum disorder,0.0
...,...,...,...,...,...,...,...
SRX7633544,SRP245232,GSE144259,Bowel,Colon,Colorectal cancer,NaN,0.0
SRX7646068,SRP245817,GSE144441,Skin,NaN,Healthy,NaN,0.0
SRX8297296,SRP192421,GSE129752,Blood,NaN,Antineutrophil cytoplasmic antibody associated...,NaN,0.0
SRX8297304,SRP192421,GSE129752,Blood,NaN,Antineutrophil cytoplasmic antibody associated...,NaN,0.0


In [7]:
# Contrastive training
tokenizer = AutoTokenizer.from_pretrained("microsoft/biogpt")
# image_processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
transcriptome_processor = GeneformerTranscriptomeProcessor(nproc=4, emb_label=model.transcriptome_model.config.emb_label)
processor = TranscriptomeTextDualEncoderProcessor(transcriptome_processor, tokenizer)

In [8]:
inputs = processor(
    text=list(adata.obs["natural_language_annotation"]),#list(adata_filt.obs["natural_language_annotation"]),
    transcriptomes=adata,#adata_filt,
    return_tensors="pt",
    padding=True,
    # max_length=128
)

/msc/home/mschae83/miniconda3/envs/singlecellm_tmpVS4/lib/python3.9/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/msc/home/mschae83/miniconda3/envs/singlecellm_tmpVS4/lib/python3.9/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/msc/home/mschae83/Varun/single-cellm/src/single_cellm/jointemb/geneformer_model.py:90: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in prepared_features.var["ensembl_id"][coding_miRNA_loc]
/msc/home/mschae83/Varun/single-cellm/src/single_cellm/jointemb/geneformer_model.py:93: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version

prepared_features has no column attribute 'filter_pass'; tokenizing all cells.


/msc/home/mschae83/Varun/single-cellm/src/single_cellm/jointemb/geneformer_model.py:142: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1682343964576/work/torch/csrc/utils/tensor_new.cpp:245.)
  tokens = torch.tensor(tokens, dtype=torch.long)


In [12]:
inputs#.keys()

{'input_ids': tensor([[  2,  56, 620,  ...,   1,   1,   1],
        [  2,  56, 620,  ...,   1,   1,   1],
        [  2,  56, 620,  ...,   1,   1,   1],
        ...,
        [  2,  56, 620,  ...,   1,   1,   1],
        [  2,  56, 620,  ...,   1,   1,   1],
        [  2,  56, 620,  ...,   1,   1,   1]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'expression_tokens': tensor([[12050, 16356,  3061,  ...,  1340,   474,  6081],
        [24388, 16145, 14483,  ...,  5584,  4858,  1667],
        [21224, 24109, 19849,  ..., 19374, 20843, 16244],
        ...,
        [14945,  2282,  7957,  ..., 10672,  1114,  8557],
        [20314, 20387, 24109,  ..., 12498, 11566, 24936],
        [20314, 20387, 24109,  ..., 10896,  6347, 20375]]), 'expression_token_lengths': tensor([2048, 2048, 2048, 2048, 2048, 2048, 20

In [8]:
adata_filt.obs

,featurecounts_Percent_Assigned_,featurecounts_Percent_Duplicates,featurecounts_Percent_Assigned_O,featurecounts_Percent_Assigned_M,featurecounts_Percent_Assigned_MO,sra_study_acc,bio_project_acc,Tissue,Tissue_subtype,Disease,...,star_number_unmapped_reads_too_many_mismatches,star_percent_unmapped_reads_too_many_mismatches,star_number_unmapped_reads_too_short,star_percent_unmapped_reads_too_short,star_number_unmapped_reads_other,star_percent_unmapped_reads_other,star_number_chimeric_reads,star_percent_chimeric_reads,has_salmon_quantification,natural_language_annotation
SRX085333,3.481,9.185,3.950,7.464,8.368,SRP007584,Rob_Research_Fusobacterium,Bowel,Colon,Colorectal cancer,...,0.0,0.0,115203.0,8.77,56325.0,4.29,0.0,0.0,False,This sample is from a primary colorectal carci...
SRX114777,1.549,9.412,1.757,9.176,10.289,SRP010181,Rob_Research_Colon,Bowel,Colon,Colorectal cancer,...,0.0,0.0,293929.0,8.42,74072.0,2.12,0.0,0.0,False,This sample is from a primary colorectal cance...
SRX099138,4.924,17.525,5.334,67.518,74.198,SRP008496,GSE32424,Esophagus,NaN,Healthy,...,0.0,0.0,869939.0,11.28,807538.0,10.47,0.0,0.0,False,"This sample is from a healthy individual, spec..."
SRX081995,18.465,41.186,20.292,31.328,40.305,SRP007412,GSE30352,Kidney,NaN,Healthy,...,0.0,0.0,2628838.0,10.33,1088758.0,4.28,0.0,0.0,False,This sample is from a healthy male kidney. It ...
SRX083168,19.856,46.335,21.700,28.790,32.158,SRP007483,GSE30573,Brain,Frontal cortex,Mental disorder,...,0.0,0.0,257888.0,0.56,136300.0,0.30,0.0,0.0,False,This sample is a primary sample of postmortem ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRX7633544,27.022,25.815,29.035,29.934,32.390,SRP245232,GSE144259,Bowel,Colon,Colorectal cancer,...,0.0,0.0,316381.0,1.00,42302.0,0.13,0.0,0.0,False,This sample is from a primary colon cancer tis...
SRX7646068,0.107,45.648,0.115,7.949,8.450,SRP245817,GSE144441,Skin,NaN,Healthy,...,0.0,0.0,10868192.0,72.09,4736.0,0.03,0.0,0.0,False,"The sample is from a healthy, non-diabetic 69-..."
SRX8297296,11.534,41.123,12.538,34.994,47.236,SRP192421,GSE129752,Blood,NaN,Antineutrophil cytoplasmic antibody associated...,...,0.0,0.0,31230.0,0.47,2428.0,0.04,0.0,0.0,False,This sample is derived from the whole blood of...
SRX8297304,21.435,36.049,23.421,40.759,47.885,SRP192421,GSE129752,Blood,NaN,Antineutrophil cytoplasmic antibody associated...,...,0.0,0.0,47139.0,0.83,5675.0,0.10,0.0,0.0,False,This sample is derived from the whole blood of...


In [9]:
inputs.keys()

dict_keys(['input_ids', 'attention_mask', 'expression_tokens', 'expression_token_lengths'])

In [10]:
inputs['expression_tokens']

tensor([[12050, 16356,  3061,  ...,  1340,   474,  6081],
        [24388, 16145, 14483,  ...,  5584,  4858,  1667],
        [21224, 24109, 19849,  ..., 19374, 20843, 16244],
        ...,
        [14945,  2282,  7957,  ..., 10672,  1114,  8557],
        [20314, 20387, 24109,  ..., 12498, 11566, 24936],
        [20314, 20387, 24109,  ..., 10896,  6347, 20375]])

### There is one last small error below


In [11]:
type(model.text_model)

transformers.models.biogpt.modeling_biogpt.BioGptModel

In [12]:
# for train and inference:
outputs = model(
    input_ids=inputs.input_ids.to(device),
    attention_mask=inputs.attention_mask.to(device),
    expression_tokens=inputs.expression_tokens.to(device),
    expression_token_lengths=inputs.expression_token_lengths.to(device),
    pad_token_id = transcriptome_processor.tokenizer.gene_token_dict.get("<pad>"),
    return_loss=True,  # TODO loss computation is still failing
)

100%|██████████| 926/926 [00:39<00:00, 23.40it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.44 GiB (GPU 0; 39.59 GiB total capacity; 37.87 GiB already allocated; 36.19 MiB free; 38.32 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
outputs.keys()

In [ ]:
print(outputs['logits_per_transcriptome'].shape)
outputs['logits_per_transcriptome']

In [ ]:
print(outputs['logits_per_text'].shape)
outputs['logits_per_text']

In [ ]:
print(outputs['text_embeds'].shape)
outputs['text_embeds']

In [ ]:
print(outputs['transcriptome_embeds'].shape)
outputs['transcriptome_embeds']

In [ ]:
print(outputs['text_model_output'].keys())
print(outputs['text_model_output']['last_hidden_state'].shape)
#outputs['text_model_output']['last_hidden_state']
print(outputs['text_model_output']['past_key_values'][0][0].shape)

In [ ]:
# Saving the model, including its configuration
model.save_pretrained("geneformer-biogpt-HVG")

In [ ]:
# # TODO needs fixed loss first (see code cell above)

# loss, logits_per_transcriptome = (
#     outputs.loss,
#     outputs.logits_per_transcriptome,
# )  # this is the transcriptome-text similarity score

# # TODO training loop

# # inference
# outputs = model(**inputs)
# logits_per_image = (
#     outputs.logits_per_transcriptome
# )  # this is the image-text similarity score
# # probs = logits_per_image.softmax(
# #     dim=1
# # )  # we can take the softmax to get the label probabilities

## Visualize Paired Embeddings

In [ ]:
import umap
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Image
pio.renderers.default = 'notebook_connected'

In [ ]:

# Assuming you have your paired embeddings ready
# For demonstration, we are using random data
text_embeddings = outputs['text_embeds'].detach().cpu().numpy()  #np.random.rand(100, 512)
transcriptome_embeddings = outputs['transcriptome_embeds'].detach().cpu().numpy() #np.random.rand(100, 512)

# Combine embeddings for UMAP
combined_embeddings = np.vstack((text_embeddings, transcriptome_embeddings))

# Perform UMAP reduction
reducer = umap.UMAP(n_components=3, random_state=42)
embedding_3d = reducer.fit_transform(combined_embeddings)

# Split the transformed embeddings back into separate arrays
text_transformed = embedding_3d[:text_embeddings.shape[0]]
transcriptome_transformed = embedding_3d[text_embeddings.shape[0]:]



In [ ]:
# Create text information for hovering from metadata
hover_text = []
for index, row in annot_metdat_df.iterrows():
    row_hover_text = "<br>".join([f"{col}: {row[col]}" for col in annot_metdat_df.drop('Text',axis=1).columns])
    hover_text.append(row_hover_text)
    
scatter_text = go.Scatter3d(
    x=text_transformed[:, 0],
    y=text_transformed[:, 1],
    z=text_transformed[:, 2],
    mode='markers',
    marker=dict(
        size=4,
        color='red',  # text points are red
    ),
    name='Text Embeddings',
    text=hover_text,  # this sets the hover text for each point
    hoverinfo='text'  # this shows the text on hover
)

scatter_transcriptome = go.Scatter3d(
    x=transcriptome_transformed[:, 0],
    y=transcriptome_transformed[:, 1],
    z=transcriptome_transformed[:, 2],
    mode='markers',
    marker=dict(
        size=4,
        color='blue',  # transcriptome points are blue
    ),
    name='Transcriptome Embeddings',
    text=hover_text,  # this sets the hover text for each point
    hoverinfo='text'  # this shows the text on hover
)

# Create line segments between paired points
lines = []
for text_point, transcriptome_point in zip(text_transformed, transcriptome_transformed):
    lines.append(
        go.Scatter3d(
            x=[text_point[0], transcriptome_point[0]],
            y=[text_point[1], transcriptome_point[1]],
            z=[text_point[2], transcriptome_point[2]],
            mode='lines',
            line=dict(
                color='grey',
                width=2,
            ),showlegend=False
        )
    )

# Combine scatter plots and lines
data = [scatter_text, scatter_transcriptome] + lines

# Configure the layout
layout = go.Layout(
    title='3D UMAP of Text and Transcriptome Embeddings',
    scene=dict(
        xaxis=dict(title='UMAP 1'),
        yaxis=dict(title='UMAP 2'),
        zaxis=dict(title='UMAP 3'),
    ),
)

# Create the visualization
fig = go.Figure(data=data, layout=layout)
# fig.show()

# For creating a GIF, you would need to repeat this process for each epoch in your training loop,
# capture the frames, and then use an image processing library like `imageio` to compile them into a GIF.


In [ ]:
fig.write_html("umap_visualization_baseline_HVG.html")

In [ ]:
# Convert the figure to a static image file and display it within Jupyter Notebook
img_bytes = pio.to_image(fig, format='png')
Image(img_bytes)